In [1]:
import sys, os
# Ensure project root is on sys.path
for p in [os.getcwd(), "/workspace/mod_gpt"]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from data.pt_dataset import (
    get_dataset, MBPPDataset, HumanEvalDataset, LiveCodeBenchDataset,
    sandbox_execute, check_code_correctness,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen3-0.6B"
print(f"Device: {DEVICE} | CWD: {os.getcwd()}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Device: cpu | CWD: /Users/fangyuanyu/Implementation/mod_gpt


## 1. Load Model & Tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(DEVICE).eval()
print(f"Model loaded: {MODEL_NAME} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: Qwen/Qwen3-0.6B (752M params)


## 2. Load All Three Datasets & Inspect Samples

In [3]:
# --- MBPP ---
mbpp_ds = get_dataset("mbpp", split="test", tokenizer=tokenizer, max_length=1024)
print(f"MBPP test: {len(mbpp_ds)} problems")
ex = mbpp_ds.dataset[0]
print(f"  Prompt: {ex['prompt'][:150]}...")
tests = mbpp_ds.get_test_cases(0)
print(f"  Tests ({len(tests)}): {tests[0][:80]}...")

# --- HumanEval ---
he_ds = get_dataset("humaneval", split="test", tokenizer=tokenizer, max_length=1024)
print(f"\nHumanEval: {len(he_ds)} problems")
ex = he_ds.dataset[0]
print(f"  Entry point: {ex['entry_point']}")
print(f"  Prompt: {ex['prompt'][:150]}...")
tests = he_ds.get_test_cases(0)
print(f"  Tests ({len(tests)}): {tests[0][:100]}...")

# --- LiveCodeBench ---
lcb_ds = get_dataset("livecodebench", split="test", tokenizer=tokenizer, max_length=1024)
print(f"\nLiveCodeBench: {len(lcb_ds)} problems")
ex = lcb_ds.dataset[0]
print(f"  Columns: {list(ex.keys())}")
question = ex.get("question_content", ex.get("question", ""))
print(f"  Question: {str(question)[:200]}...")
tests = lcb_ds.get_test_cases(0)
print(f"  Tests ({len(tests)}): {type(tests[0]) if tests else 'none'}")

MBPP test: 257 problems
  Prompt: Write a python function to remove first and last occurrence of a given character from the string....
  Tests (3): assert remove_Occ("hello","l") == "heo"...

HumanEval: 164 problems
  Entry point: has_close_elements
  Prompt: from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any t...
  Tests (1): 

METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candid...

LiveCodeBench: 1055 problems
  Columns: ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata']
  Question: There are three cards with letters $\texttt{a}$, $\texttt{b}$, $\texttt{c}$ placed in a row in some order. You can do the following operation at most once: 

 
-  Pick two cards, and swap them.  Is it...
  Tests (1): <class 'dict'>


In [ ]:
## 3. Sanity Check: Ground-Truth Solutions Pass Their Own Tests\n\nVerify the sandbox works by running **reference solutions** against their tests.\n- MBPP: has `code` field with solutions\n- HumanEval: has `canonical_solution` field\n- LiveCodeBench: no reference solutions (skip)

## was: Sanity Check section header (merged into cell 6)

In [ ]:
import json

# --- MBPP: ground-truth check ---
print("=== MBPP: Ground-truth solution check ===")
for i in range(5):
    ex = mbpp_ds.dataset[i]
    code = ex["code"]
    tests = mbpp_ds.get_test_cases(i)
    result = check_code_correctness(code, tests, timeout=10)
    status = "✅" if result["passed"] else f"❌ ({result['errors'][:1]})"
    print(f"  Problem {i}: {status}  ({result['num_passed']}/{result['num_total']} tests)")

# --- HumanEval: ground-truth check ---
print("\n=== HumanEval: Ground-truth solution check ===")
for i in range(5):
    ex = he_ds.dataset[i]
    # HumanEval: prompt contains the function signature, canonical_solution is the body
    code = ex["prompt"] + ex["canonical_solution"]
    tests = he_ds.get_test_cases(i)
    result = check_code_correctness(code, tests, timeout=10)
    status = "✅" if result["passed"] else f"❌ ({result['errors'][:1]})"
    print(f"  Problem {i} ({ex['entry_point']}): {status}")

# --- LiveCodeBench: just verify test case parsing ---
print("\n=== LiveCodeBench: Test case parsing check ===")
lcb_with_tests = 0
for i in range(min(50, len(lcb_ds))):
    tests = lcb_ds.get_test_cases(i)
    if tests:
        lcb_with_tests += 1
        if lcb_with_tests <= 3:
            print(f"  Problem {i}: {len(tests)} test cases found")
print(f"  {lcb_with_tests}/50 problems have parseable test cases")

## 4. Generate Code with Qwen3-0.6B\n\nTest model generation + execution on a few problems from each benchmark.

In [ ]:
import pprint
from concurrent.futures import ProcessPoolExecutor, as_completed

@torch.no_grad()
def generate_code(model, tokenizer, prompt_text, max_new_tokens=256):
    """Generate code from a single prompt (kept for run_demo)."""
    inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


@torch.no_grad()
def generate_code_batch(model, tokenizer, prompts, max_new_tokens=256):
    """Batched generation: left-pad prompts, generate, decode per-sample new tokens."""
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    ).to(DEVICE)
    prompt_lens = enc["attention_mask"].sum(dim=1)  # real token counts per sample

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Decode only the newly generated tokens for each sample
    results = []
    for i in range(len(prompts)):
        new_tokens = out[i, prompt_lens[i]:]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return results


def build_prompt(dataset, idx):
    """Build the generation prompt for a given dataset and index."""
    ex = dataset.dataset[idx]
    if isinstance(dataset, MBPPDataset):
        return f"# Task: {ex['prompt'].strip()}\n# Solution:\n"
    elif isinstance(dataset, HumanEvalDataset):
        return ex["prompt"]
    else:  # LiveCodeBench / CodeContests
        question = ex.get("question_content", ex.get("question", "")).strip()
        starter = ex.get("starter_code", "").strip()
        if starter:
            return f"# Problem:\n{question}\n\n{starter}\n# Solution:\n"
        return f"# Problem:\n{question}\n\n# Solution:\n"


def run_demo(dataset, name, n=3, max_new_tokens=256):
    """Generate + execute on a few problems. Show full check_code_correctness result."""
    print(f"{'='*60}")
    print(f"  {name}: Generate -> Execute -> check_code_correctness")
    print(f"{'='*60}\n")
    tested = 0
    for i in range(min(100, len(dataset))):
        tests = dataset.get_test_cases(i)
        if not tests:
            continue
        prompt = build_prompt(dataset, i)
        generated = generate_code(model, tokenizer, prompt, max_new_tokens=max_new_tokens)
        if isinstance(dataset, HumanEvalDataset):
            exec_code = prompt + generated
        else:
            exec_code = generated
        result = check_code_correctness(exec_code, tests, timeout=10)
        status = "PASS" if result["passed"] else "FAIL"
        print(f"--- Problem {i}: {status} ---")
        print(f"Prompt:\n{prompt[:200]}...\n")
        print(f"Generated code:\n```python\n{generated[:500]}\n```\n")
        print(f"Test cases ({len(tests)}):")
        for j, t in enumerate(tests[:3]):
            if isinstance(t, str):
                print(f"  [{j}] {t[:120]}")
            else:
                print(f"  [{j}] preamble: {t['preamble'][:60]}... | check: {t['check'][:60]}...")
        if len(tests) > 3:
            print(f"  ... +{len(tests)-3} more")
        print(f"\ncheck_code_correctness result:")
        pprint.pprint(result, width=100)
        if not result["passed"] and result["errors"]:
            print(f"\nError details:")
            for j, err in enumerate(result["errors"][:3]):
                print(f"  test[{j}]: {err[:300] if err else 'None'}")
        print("\n")
        tested += 1
        if tested >= n:
            break


run_demo(mbpp_ds, "MBPP", n=5, max_new_tokens=256)

In [ ]:
run_demo(he_ds, "HumanEval", n=5, max_new_tokens=256)

In [ ]:
run_demo(lcb_ds, "LiveCodeBench", n=5, max_new_tokens=512)

In [ ]:
## 5. Batch Eval: Pass@1 on N samples\n\nRun a proper pass@1 evaluation loop on a small subset of each benchmark.

In [ ]:
# Q1. why is there a specific logic for human eval? what differentiates human eval, and what's the same between LCB & MBPP?
#     how come run_demo is uniform for all datasets? 

# A1. It appears that in HumanEval, we do function completion given its header, whilst in MBPP and LCB, we need to generate the entire function
#     including the decorator and input arguments etc.
# if isinstance(dataset, HumanEvalDataset):
#            exec_code = prompt + generated
#        else:
#            exec_code = generated

dataset = mbpp_ds
i = 0 
max_new_tokens = 256
prompt = build_prompt(dataset, i)
generated = generate_code(model, tokenizer, prompt, max_new_tokens=max_new_tokens)

indices, prompts, test_cases_list = [], [], []
for i in range(5): 
    indices.append(i)
    prompts.append(build_prompt(dataset, i))
    test_cases_list.append(tests)

batch_gen = generate_code_batch(model, tokenizer, prompts, max_new_tokens=max_new_tokens)

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def eval_pass_at_1(model, tokenizer, dataset, n_samples=20, max_new_tokens=256, 
                   timeout=10, batch_size=8):
    """Batched pass@1 evaluation with execution-based checking."""
    is_humaneval = isinstance(dataset, HumanEvalDataset)
    
    # Collect problems that have test cases
    indices, prompts, test_cases_list = [], [], []
    for i in range(min(n_samples * 3, len(dataset))):
        tests = dataset.get_test_cases(i)
        if tests:
            indices.append(i)
            prompts.append(build_prompt(dataset, i))
            test_cases_list.append(tests)
        if len(indices) >= n_samples:
            break
    
    tested = len(indices)
    skipped = min(n_samples * 3, len(dataset)) - tested  # approximate
    passed = 0
    
    # Batched generation
    all_generated = []
    for b_start in tqdm(range(0, tested, batch_size), desc="Generating"):
        b_end = min(b_start + batch_size, tested)
        batch_prompts = prompts[b_start:b_end]
        batch_gen = generate_code_batch(model, tokenizer, batch_prompts, max_new_tokens=max_new_tokens)
        all_generated.extend(batch_gen)
    
    # Evaluation (sandbox is CPU-bound, can run in parallel)
    def _check_one(args):
        idx_in_batch, gen_text, prompt_text, tests = args
        if is_humaneval:
            exec_code = prompt_text + gen_text
        else:
            exec_code = gen_text
        return check_code_correctness(exec_code, tests, timeout=timeout)
    
    results = []
    # Use threads for parallel sandbox execution
    from concurrent.futures import ThreadPoolExecutor
    eval_args = [(j, all_generated[j], prompts[j], test_cases_list[j]) for j in range(tested)]
    with ThreadPoolExecutor(max_workers=min(8, tested)) as pool:
        results = list(tqdm(pool.map(lambda a: _check_one(a), eval_args), 
                           total=tested, desc="Evaluating"))
    
    passed = sum(1 for r in results if r["passed"])
    acc = passed / max(tested, 1)
    print(f"\npass@1 = {passed}/{tested} = {acc:.1%}  (skipped ~{skipped} w/o tests)")
    return {"pass_at_1": acc, "passed": passed, "tested": tested, "skipped": skipped}


N = 20  # samples per benchmark
BS = 8   # generation batch size

print("=" * 60)
print(f"MBPP pass@1 ({N} samples, batch_size={BS}) — in-distribution")
print("=" * 60)
mbpp_result = eval_pass_at_1(model, tokenizer, mbpp_ds, n_samples=N, 
                              max_new_tokens=256, batch_size=BS)

print()
print("=" * 60)
print(f"HumanEval pass@1 ({N} samples, batch_size={BS}) — OOD eval")
print("=" * 60)
he_result = eval_pass_at_1(model, tokenizer, he_ds, n_samples=N, 
                            max_new_tokens=256, batch_size=BS)

print()
print("=" * 60)
print(f"LiveCodeBench pass@1 ({N} samples, batch_size={BS}) — OOD eval")
print("=" * 60)
lcb_result = eval_pass_at_1(model, tokenizer, lcb_ds, n_samples=N, 
                             max_new_tokens=512, batch_size=BS)